# QIIME2 16S Amplicon Workflow

**Tier 3 — Applied Bioinformatics | Module 26 · Notebook 4**

*Prerequisites: Module 04 (Microbial Diversity), Notebook 1 (Taxonomic Profiling)*

---

**By the end of this notebook you will be able to:**
1. Import paired-end 16S amplicon reads into QIIME2 artifacts
2. Denoise reads with DADA2 to produce an ASV feature table
3. Assign taxonomy to ASVs using a pre-trained classifier
4. Compute alpha and beta diversity metrics
5. Identify differentially abundant taxa with ANCOM-BC


**Key resources:**
- [QIIME2 Moving Pictures tutorial](https://docs.qiime2.org/2024.10/tutorials/moving-pictures/)
- [QIIME2 documentation](https://docs.qiime2.org/)
- [Galaxy Training — 16S Metagenomics](https://training.galaxyproject.org/training-material/topics/metagenomics/)

---[< Previous: Metagenomic Assembly, Binning & MAGs](03_assembly_binning.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: Data Harmonization for Multi-Omics >](../27_Multi_Omics_Integration/01_data_harmonization.ipynb)---

## 1. Import Reads into QIIME2

**QIIME2** (Quantitative Insights Into Microbial Ecology 2) is a comprehensive amplicon sequencing analysis framework. All data in QIIME2 is stored as **artifacts** (`.qza` files) with provenance tracking, or **visualizations** (`.qzv` files) viewable at https://view.qiime2.org.

### Manifest Format (for paired-end reads)

The manifest CSV file maps sample IDs to absolute paths of their FASTQ files:

```csv
sample-id,absolute-filepath,direction
Sample1,/data/Sample1_R1.fastq.gz,forward
Sample1,/data/Sample1_R2.fastq.gz,reverse
Sample2,/data/Sample2_R1.fastq.gz,forward
Sample2,/data/Sample2_R2.fastq.gz,reverse
```

```bash
# Import using manifest file
qiime tools import \
    --type 'SampleData[PairedEndSequencesWithQuality]' \
    --input-path manifest.csv \
    --output-path reads.qza \
    --input-format PairedEndFastqManifestPhred33

# Check quality to determine DADA2 truncation lengths
qiime demux summarize \
    --i-data reads.qza \
    --o-visualization reads_summary.qzv
# Open reads_summary.qzv at view.qiime2.org to see per-position quality boxplots
```

**Key decision from quality visualization:** Identify position where median quality drops below Q20 — use this as the truncation length for DADA2.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# Simulate per-position quality scores (as would be seen in QIIME2 demux summarize)
read_length = 250  # V3-V4 region, 250 bp paired-end
n_reads_vis = 5000
positions = np.arange(1, read_length + 1)

# Forward reads: quality drops in last ~30 bp
qual_mean_fwd = 37 - 0.03 * positions - 3 * np.exp(-(read_length - positions) / 30)
qual_q25_fwd  = qual_mean_fwd - 2 - 0.01 * positions
qual_q75_fwd  = qual_mean_fwd + 1.5

# Reverse reads: quality drops faster (typical paired-end behavior)
qual_mean_rev = 36 - 0.06 * positions - 5 * np.exp(-(read_length - positions) / 20)
qual_q25_rev  = qual_mean_rev - 3 - 0.02 * positions
qual_q75_rev  = qual_mean_rev + 1.5

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (mean_q, q25, q75, label) in zip(axes, [
    (qual_mean_fwd, qual_q25_fwd, qual_q75_fwd, 'Forward reads'),
    (qual_mean_rev, qual_q25_rev, qual_q75_rev, 'Reverse reads'),
]):
    ax.fill_between(positions, np.clip(q25, 0, 40), np.clip(q75, 0, 40),
                    alpha=0.3, color='steelblue', label='Q25–Q75')
    ax.plot(positions, np.clip(mean_q, 0, 40), color='steelblue', lw=1.5, label='Median Q')
    ax.axhline(20, color='red',    linestyle='--', lw=1, label='Q20 threshold')
    ax.axhline(30, color='orange', linestyle=':', lw=1, label='Q30 threshold')
    # Suggest truncation at position where median drops below Q25
    trunc_pos = positions[np.where(mean_q < 25)[0][0]] if (mean_q < 25).any() else read_length
    ax.axvline(trunc_pos, color='green', linestyle='-', lw=2, alpha=0.7,
               label=f'Truncate at pos {trunc_pos}')
    ax.set_xlabel('Position (bp)')
    ax.set_ylabel('Phred quality score')
    ax.set_title(f'QIIME2 demux summary: {label}')
    ax.set_xlim(1, read_length)
    ax.set_ylim(0, 42)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()
print('Recommended DADA2 truncation: --p-trunc-len-f 220 --p-trunc-len-r 200')


## 2. Denoising with DADA2

**DADA2** (Divisive Amplicon Denoising Algorithm 2) corrects sequencing errors and produces **Amplicon Sequence Variants (ASVs)** — exact sequence variants at single-nucleotide resolution — rather than the traditional OTUs (Operational Taxonomic Units) clustered at 97% similarity.

**ASV vs OTU:**
- OTU: sequences clustered at 97% similarity threshold; single representative per cluster; some diversity lost
- ASV: exact error-corrected sequences; no clustering; two ASVs can differ by a single nucleotide; reproducible across studies (same ASV = same sequence in any study)

```bash
# DADA2 denoising (paired-end)
qiime dada2 denoise-paired \
    --i-demultiplexed-seqs reads.qza \
    --p-trim-left-f 13 \          # trim primer/adapter from 5' end of forward reads
    --p-trim-left-r 13 \          # trim primer/adapter from 5' end of reverse reads
    --p-trunc-len-f 220 \         # truncate forward reads at position 220
    --p-trunc-len-r 200 \         # truncate reverse reads at position 200
    --p-n-threads 0 \             # 0 = use all available cores
    --o-table feature_table.qza \ # ASV feature table (samples × ASVs)
    --o-representative-sequences rep_seqs.qza \  # ASV sequences
    --o-denoising-stats stats.qza
```

**Choosing truncation lengths:** Forward and reverse reads must overlap by ≥20 bp after truncation for merging. For V3-V4 (expected amplicon ~460 bp): using 220 + 200 gives 420 bp combined, which overlaps sufficiently on a 460 bp amplicon.

**Quality check:** After denoising, visualize stats to ensure ≥75% of input reads pass filtering and ≥60% are merged successfully.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)

# Simulate DADA2 denoising statistics across samples
n_samples = 16
sample_names = [f'S{i+1:02d}' for i in range(n_samples)]

input_reads    = rng.integers(20_000, 80_000, size=n_samples)
filtered_pct   = rng.uniform(0.82, 0.95, size=n_samples)   # 82-95% pass filtering
denoised_pct   = rng.uniform(0.90, 0.99, size=n_samples)   # 90-99% denoised
merged_pct     = rng.uniform(0.78, 0.92, size=n_samples)   # 78-92% merged
non_chimeric   = rng.uniform(0.93, 0.99, size=n_samples)   # 93-99% non-chimeric

filtered   = (input_reads * filtered_pct).astype(int)
denoised   = (filtered * denoised_pct).astype(int)
merged     = (denoised * merged_pct).astype(int)
final      = (merged * non_chimeric).astype(int)

stats_df = pd.DataFrame({
    'Sample':     sample_names,
    'Input':      input_reads,
    'Filtered':   filtered,
    'Denoised':   denoised,
    'Merged':     merged,
    'Non-chimeric': final,
    'Retention_%': final / input_reads * 100,
})

print('DADA2 denoising statistics (first 8 samples):')
print(stats_df.head(8)[['Sample', 'Input', 'Filtered', 'Merged', 'Non-chimeric', 'Retention_%']].to_string(index=False))
print(f'\nMean overall retention: {(final / input_reads).mean():.1%}')

# Simulate ASV counts
n_asvs_per_sample = rng.integers(200, 800, size=n_samples)
print(f'Mean ASVs per sample: {n_asvs_per_sample.mean():.0f} (range: {n_asvs_per_sample.min()}–{n_asvs_per_sample.max()})')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Read retention waterfall
steps = ['Input', 'Filtered', 'Merged', 'Non-chimeric']
step_means = [input_reads.mean(), filtered.mean(), merged.mean(), final.mean()]
axes[0].bar(steps, step_means, color=['#4C72B0', '#55A868', '#DD8452', '#C44E52'], alpha=0.85)
axes[0].set_ylabel('Mean reads per sample')
axes[0].set_title('DADA2 read retention pipeline')
for i, (step, val) in enumerate(zip(steps, step_means)):
    axes[0].text(i, val + 500, f'{val:,.0f}', ha='center', fontsize=9)

# Retention % distribution
axes[1].hist(stats_df['Retention_%'], bins=10, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(75, color='red', linestyle='--', lw=1.5, label='75% threshold')
axes[1].set_xlabel('Overall read retention (%)')
axes[1].set_ylabel('Number of samples')
axes[1].set_title('DADA2 read retention distribution')
axes[1].legend()

plt.tight_layout()
plt.show()


## 3. Taxonomic Classification with SILVA

**SILVA** (silva.nr_v138) is the standard reference database for 16S rRNA classification in QIIME2. Pre-trained Naive Bayes classifiers are available from the QIIME2 data resources page — use the classifier trained for your specific primer pair and read length.

```bash
# Download pre-trained SILVA 138 classifier (V3-V4, 515F/806R primers)
# Available at: https://data.qiime2.org/2024.5/common/silva-138-99-seqs-515-806.qza

# Classify ASVs
qiime feature-classifier classify-sklearn \
    --i-classifier silva138_nb_515_806_classifier.qza \
    --i-reads rep_seqs.qza \
    --o-classification taxonomy.qza \
    --p-n-jobs 1

# Generate taxonomy bar chart
qiime taxa barplot \
    --i-table feature_table.qza \
    --i-taxonomy taxonomy.qza \
    --m-metadata-file metadata.tsv \
    --o-visualization taxa_barplot.qzv

# Filter out mitochondria and chloroplast sequences (contaminants from host plant/food)
qiime taxa filter-table \
    --i-table feature_table.qza \
    --i-taxonomy taxonomy.qza \
    --p-exclude Mitochondria,Chloroplast,Eukaryota \
    --o-filtered-table feature_table_filtered.qza
```

**Confidence threshold:** The `--p-confidence 0.7` default means assignments below 70% bootstrap confidence are reported as unclassified at that taxonomic level. For species-level classification, accuracy drops substantially; genus-level is more reliable.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

rng = np.random.default_rng(20)

# Simulate taxonomy classification for 16 gut metagenome samples
taxa = {
    'g__Bacteroides':       (0.18, 0.08),
    'g__Faecalibacterium':  (0.10, 0.06),
    'g__Prevotella':        (0.08, 0.07),
    'g__Bifidobacterium':   (0.07, 0.04),
    'g__Ruminococcus':      (0.06, 0.04),
    'g__Akkermansia':       (0.05, 0.04),
    'g__Blautia':           (0.05, 0.03),
    'g__Lachnospiraceae':   (0.06, 0.04),
    'g__Streptococcus':     (0.04, 0.03),
    'g__Clostridium':       (0.04, 0.03),
    'Other':                (0.27, 0.05),
}

n_samples = 16
conditions = ['Healthy'] * 8 + ['IBD'] * 8
sample_names = [f'{c[:1]}{i+1}' for i, c in enumerate(conditions)]

n_taxa = len(taxa)
compositions = np.zeros((n_samples, n_taxa))
for j, (genus, (mean, std)) in enumerate(taxa.items()):
    base = rng.normal(mean, std, n_samples)
    # Reduce Faecalibacterium in IBD (known association)
    if genus == 'g__Faecalibacterium':
        base[8:] *= 0.4
    compositions[:, j] = np.clip(base, 0.001, 1)

# Re-normalize to sum to 1
compositions = compositions / compositions.sum(axis=1, keepdims=True)
comp_df = pd.DataFrame(compositions, columns=list(taxa.keys()), index=sample_names)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Stacked bar chart (taxonomy bar plot)
colors = cm.tab20(np.linspace(0, 1, n_taxa))
bottom = np.zeros(n_samples)
for j, (genus, col) in enumerate(zip(list(taxa.keys()), colors)):
    vals = comp_df.iloc[:, j].values
    axes[0].bar(range(n_samples), vals, bottom=bottom, label=genus, color=col, width=0.85)
    bottom += vals

axes[0].set_xticks(range(n_samples))
axes[0].set_xticklabels(sample_names, rotation=45, ha='right', fontsize=8)
axes[0].set_ylabel('Relative abundance')
axes[0].set_title('QIIME2 taxonomy bar chart (genus level)')
axes[0].legend(loc='upper right', fontsize=6, ncol=2)
# Add condition separator
axes[0].axvline(7.5, color='black', lw=2, linestyle='--')
axes[0].text(3.5, 1.02, 'Healthy', ha='center', fontsize=9, color='green')
axes[0].text(11.5, 1.02, 'IBD',    ha='center', fontsize=9, color='red')

# Faecalibacterium comparison
fpraus = comp_df['g__Faecalibacterium']
cond_colors = ['#55A868'] * 8 + ['#C44E52'] * 8
axes[1].bar(range(n_samples), fpraus.values, color=cond_colors, alpha=0.8, edgecolor='white')
axes[1].set_xticks(range(n_samples))
axes[1].set_xticklabels(sample_names, rotation=45, ha='right', fontsize=8)
axes[1].axvline(7.5, color='black', lw=1.5, linestyle='--')
axes[1].set_ylabel('Relative abundance')
axes[1].set_title('Faecalibacterium prausnitzii\n(reduced in IBD)')

from matplotlib.patches import Patch
axes[1].legend(handles=[Patch(color='#55A868', label='Healthy'),
                         Patch(color='#C44E52', label='IBD')])

plt.tight_layout()
plt.show()


## 4. Diversity Analysis

### Phylogenetic Tree Construction

Alpha and beta diversity metrics that account for evolutionary relationships (Faith's PD, UniFrac) require a phylogenetic tree of the ASV sequences.

```bash
# Multiple sequence alignment (MAFFT) and tree (FastTree)
qiime phylogeny align-to-tree-mafft-fasttree \
    --i-sequences rep_seqs.qza \
    --o-alignment aligned_rep_seqs.qza \
    --o-masked-alignment masked_aligned.qza \
    --o-tree unrooted_tree.qza \
    --o-rooted-tree rooted_tree.qza \
    --p-n-threads 8
```

### Core Diversity Metrics

```bash
qiime diversity core-metrics-phylogenetic \
    --i-phylogeny rooted_tree.qza \
    --i-table feature_table_filtered.qza \
    --p-sampling-depth 5000 \    # rarefaction depth — samples below this are dropped
    --m-metadata-file metadata.tsv \
    --output-dir diversity_metrics/
```

**Alpha diversity metrics produced:**
- `faith_pd_vector.qza`: Faith's Phylogenetic Diversity (sum of branch lengths of the ASVs present)
- `shannon_vector.qza`: Shannon entropy H = -Σ p_i log(p_i)
- `observed_features_vector.qza`: raw ASV richness
- `evenness_vector.qza`: Pielou's J = Shannon / log(richness)

**Beta diversity metrics:**
- `bray_curtis_distance_matrix.qza`: abundance-weighted dissimilarity (no phylogeny)
- `unweighted_unifrac_distance_matrix.qza`: phylogeny-informed presence/absence
- `weighted_unifrac_distance_matrix.qza`: phylogeny-informed + abundance-weighted

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import braycurtis
from sklearn.decomposition import PCA
from scipy.stats import entropy

rng = np.random.default_rng(30)

# Simulate feature table post-DADA2 (samples x ASVs)
n_samples = 16
n_asvs    = 400
conditions = ['Healthy'] * 8 + ['IBD'] * 8
sample_names = [f'{c[:1]}{i+1}' for i, c in enumerate(conditions)]

# Simulate composition (Healthy has higher diversity)
alpha_healthy = rng.exponential(1.5, n_asvs) + 0.2
alpha_ibd     = rng.exponential(0.5, n_asvs) + 0.1   # lower alpha

table = np.zeros((n_samples, n_asvs))
for i in range(n_samples):
    alpha = alpha_healthy if conditions[i] == 'Healthy' else alpha_ibd
    props = rng.dirichlet(alpha)
    # Rarefy to 5000 reads
    table[i] = rng.multinomial(5000, props)

# Alpha diversity
shannon_vals = np.array([entropy(table[i][table[i] > 0]) for i in range(n_samples)])
richness = (table > 0).sum(axis=1)

# Beta diversity: Bray-Curtis distance matrix
bc_matrix = np.zeros((n_samples, n_samples))
for i in range(n_samples):
    for j in range(n_samples):
        bc_matrix[i, j] = braycurtis(table[i], table[j])

# PCoA on BC distances
from sklearn.manifold import MDS
mds = MDS(n_components=2, dissimilarity='precomputed', random_state=42)
pcoa_coords = mds.fit_transform(bc_matrix)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Alpha diversity boxplot
cond_colors = {'Healthy': '#55A868', 'IBD': '#C44E52'}
for cond, col in cond_colors.items():
    mask = [c == cond for c in conditions]
    axes[0].scatter(np.where(mask)[0], shannon_vals[mask], color=col, s=60, label=cond, zorder=3)
    axes[0].hlines(shannon_vals[mask].mean(), np.where(mask)[0].min(), np.where(mask)[0].max(),
                   colors=col, linestyles='--', lw=1.5)
axes[0].set_xticks(range(n_samples))
axes[0].set_xticklabels(sample_names, rotation=45, ha='right', fontsize=7)
axes[0].set_ylabel('Shannon diversity')
axes[0].set_title('Alpha diversity (Shannon)')
axes[0].legend()

# PCoA (beta diversity)
for cond, col in cond_colors.items():
    mask = [c == cond for c in conditions]
    axes[1].scatter(pcoa_coords[mask, 0], pcoa_coords[mask, 1],
                    color=col, s=70, label=cond, alpha=0.8)
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].set_title('PCoA of Bray-Curtis dissimilarity')
axes[1].legend()

# Distance heatmap
import seaborn as sns
col_colors = ['#55A868' if c == 'Healthy' else '#C44E52' for c in conditions]
cg = sns.heatmap(bc_matrix, ax=axes[2], cmap='YlOrRd',
                  xticklabels=sample_names, yticklabels=sample_names,
                  cbar_kws={'label': 'Bray-Curtis distance'})
axes[2].set_title('Beta diversity: Bray-Curtis matrix')
axes[2].tick_params(axis='both', labelsize=7)

plt.tight_layout()
plt.show()
print(f'Mean Shannon: Healthy={shannon_vals[:8].mean():.2f}, IBD={shannon_vals[8:].mean():.2f}')


## 5. Differential Abundance with ANCOM-BC

Standard statistical tests cannot be directly applied to relative abundance data from sequencing because the data are **compositional** — they sum to a constant (total read depth). A real increase in one taxon necessarily causes apparent decreases in others. **ANCOM-BC** (Analysis of Compositions of Microbiomes with Bias Correction) addresses this by:
1. Estimating and correcting for sample-specific sampling fractions (the log ratio of observed to true abundance)
2. Applying a log-ratio model to detect true differential abundance

```bash
qiime composition ancombc \
    --i-table feature_table_filtered.qza \
    --m-metadata-file metadata.tsv \
    --p-formula 'condition' \
    --o-differentials differentials.qza

qiime composition da-barplot \
    --i-data differentials.qza \
    --p-significance-threshold 0.05 \
    --o-visualization da_barplot.qzv
```

**Alternative: LEfSe** (Linear discriminant analysis Effect Size) — still widely used but does not properly handle compositional structure. ANCOM-BC is preferred for rigorous analysis.

**ANCOM-BC output:**
- `lfc`: log-fold change (condition B vs A)
- `se`: standard error
- `W`: test statistic
- `p_val`, `q_val`: p-value and BH-adjusted q-value

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests

rng = np.random.default_rng(88)

# Simulate ANCOM-BC differential abundance results
# 200 ASVs tested across Healthy vs IBD
n_asvs = 200
taxa_labels = [f'ASV_{i:03d}' for i in range(n_asvs)]

# Assign rough genus to top ASVs
genus_map = {
    'ASV_000': 'Faecalibacterium', 'ASV_001': 'Akkermansia',
    'ASV_002': 'Bacteroides',       'ASV_003': 'Prevotella',
    'ASV_004': 'Ruminococcus',      'ASV_005': 'Bifidobacterium',
    'ASV_006': 'Enterococcus',      'ASV_007': 'Escherichia',
}

# Simulate log-fold changes and standard errors
lfc = rng.normal(0, 0.6, n_asvs)
# Force biological signal: Faecalibacterium/Akkermansia/Bifidobacterium down in IBD
for asv, genus in genus_map.items():
    idx = int(asv.split('_')[1])
    if genus in ('Faecalibacterium', 'Akkermansia', 'Bifidobacterium'):
        lfc[idx] = rng.normal(-1.8, 0.3)
    elif genus in ('Escherichia', 'Enterococcus'):
        lfc[idx] = rng.normal(1.5, 0.3)

se = np.abs(rng.normal(0.3, 0.1, n_asvs)) + 0.15
w_stat = lfc / se
pvals  = stats.norm.sf(np.abs(w_stat)) * 2
_, qvals, _, _ = multipletests(pvals, method='BH')

ancom_df = pd.DataFrame({
    'ASV':   taxa_labels,
    'Genus': [genus_map.get(a, 'Other') for a in taxa_labels],
    'LFC':   lfc,
    'SE':    se,
    'W':     w_stat,
    'pval':  pvals,
    'qval':  qvals,
}).sort_values('qval')

sig = ancom_df[ancom_df['qval'] < 0.05]
print(f'Significant ASVs (FDR < 5%): {len(sig)}')
print(sig[['ASV', 'Genus', 'LFC', 'qval']].head(10).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Volcano plot
sig_mask = ancom_df['qval'] < 0.05
colors_v = np.where(sig_mask & (ancom_df['LFC'] > 0), '#C44E52',
           np.where(sig_mask & (ancom_df['LFC'] < 0), '#4C72B0', '#AAAAAA'))
axes[0].scatter(ancom_df['LFC'], -np.log10(ancom_df['pval']),
                c=colors_v, s=20, alpha=0.7)
axes[0].axhline(-np.log10(0.05), color='orange', linestyle=':', lw=1.5, label='p=0.05')
axes[0].axvline(0, color='grey', linestyle='--', lw=1)
# Label known genera
for _, row in sig.head(8).iterrows():
    if row['Genus'] != 'Other':
        axes[0].annotate(row['Genus'], (row['LFC'], -np.log10(row['pval'])),
                         xytext=(5, 3), textcoords='offset points', fontsize=7)
axes[0].set_xlabel('Log-fold change (IBD vs Healthy)')
axes[0].set_ylabel('-log10(p-value)')
axes[0].set_title('ANCOM-BC: differential ASVs')
axes[0].legend(handles=[
    plt.scatter([], [], c='#C44E52', label=f'Enriched in IBD ({(sig["LFC"]>0).sum()})'),
    plt.scatter([], [], c='#4C72B0', label=f'Depleted in IBD ({(sig["LFC"]<0).sum()})'),
], fontsize=8)

# DA barplot (top significant ASVs)
top_sig = sig.head(12).sort_values('LFC')
bar_colors = ['#C44E52' if l > 0 else '#4C72B0' for l in top_sig['LFC']]
axes[1].barh(range(len(top_sig)), top_sig['LFC'], color=bar_colors, alpha=0.85)
axes[1].set_yticks(range(len(top_sig)))
labels = [f"{row['ASV']} ({row['Genus']})" if row['Genus'] != 'Other' else row['ASV']
          for _, row in top_sig.iterrows()]
axes[1].set_yticklabels(labels, fontsize=8)
axes[1].axvline(0, color='black', lw=1)
axes[1].set_xlabel('Log-fold change (IBD vs Healthy)')
axes[1].set_title('Top differential ASVs (ANCOM-BC)')

plt.tight_layout()
plt.show()


## Summary

### QIIME2 16S Amplicon Workflow

1. **Import**: manifest CSV → QIIME2 artifact; inspect quality with `demux summarize`
2. **DADA2 denoising**: trim primers → truncate based on quality → error correction → ASV table + representative sequences
3. **Taxonomic classification**: Naive Bayes classifier against SILVA 138 → taxonomy artifact; filter out mitochondria/chloroplast
4. **Phylogenetic tree**: MAFFT + FastTree → rooted tree (required for UniFrac and Faith's PD)
5. **Core diversity**: rarefaction → alpha diversity (Shannon, PD) + beta diversity (Bray-Curtis, UniFrac) + PCoA
6. **Differential abundance**: ANCOM-BC → compositional-aware LFC estimation with bias correction

### ASV vs OTU

| | OTU (97%) | ASV |
|---|---|---|
| Resolution | Genus level typical | Strain level possible |
| Reproducibility | Database-dependent | Exact sequence (cross-study reproducible) |
| Chimera handling | In clustering | Removed by DADA2 explicitly |
| Standard tool | VSEARCH, UCLUST | DADA2, Deblur |

### 16S vs Shotgun Comparison

16S amplicon sequencing (this notebook) provides cost-effective community profiling at the genus level with well-established pipelines. Shotgun metagenomics (Notebooks 1-3) offers species/strain resolution, functional information, and AMR gene content but at higher cost and bioinformatic complexity.

For the shotgun functional annotation workflow, see: [Notebook 2: Functional Annotation](./02_functional_annotation.ipynb)

---[< Previous: Metagenomic Assembly, Binning & MAGs](03_assembly_binning.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: Data Harmonization for Multi-Omics >](../27_Multi_Omics_Integration/01_data_harmonization.ipynb)---